In [ ]:
import pandas as pd
vcc_data = pd.read_csv('/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/who_vaccine/H3N2_vaccine_sequences.csv')
vcc_seqs = {
    str(name).strip(): str(seq).strip().upper() 
    for name, seq in zip(vcc_data['Vaccines_name'], vcc_data['sequence'])
}
print(len(vcc_seqs))
print(vcc_seqs.keys())

def read_custom_fasta(path):
    sequences = {}
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 2):
        name = lines[i][1:]
        seq = lines[i + 1].upper()
        sequences[name] = seq.upper()

    return sequences 

h3_68_03_seqs = read_custom_fasta('/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2_smith.fasta')
print(len(h3_68_03_seqs))
h3_03_24_seqs = read_custom_fasta('/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2.fasta')
print(len(h3_03_24_seqs))

24
dict_keys(['A/Croatia/10136RV/2023|EPI_ISL_19296516', 'A/Thailand/8/2022|EPI_ISL_16014504', 'A/Darwin/9/2021|EPI_ISL_2233240', 'A/Cambodia/e0826360/2020|EPI_ISL_806547', 'A/Hong Kong/2671/2019|EPI_ISL_20140890', 'A/Kansas/14/2017|EPI_ISL_363642', 'A/Singapore/INFIMH-16-0019/2016|EPI_ISL_291274', 'A/Hong Kong/4801/2014|EPI_ISL_20140890', 'A/Switzerland/9715293/2013|EPI_ISL_198223', 'A/Texas/50/2012|EPI_ISL_129744', 'A/Victoria/361/2011|EPI_ISL_104004', 'A/Perth/16/2009|EPI_ISL_60761', 'A/Brisbane/10/2007|EPI_ISL_177756', 'A/Wisconsin/67/2005|KM821341.1', 'A/California/7/2004|EPI_ISL_113070', 'A/Fujian/411/2002|EPI_ISL_107711', 'A/Moscow/10/99|DQ487341.1', 'A/Sydney/5/97|EPI_ISL_1313', 'A/South Australia/34/2019|EPI_ISL_20141035', 'A/Switzerland/8060/2017|EPI_ISL_330887', 'A/Singapore/INFIMH-16-0019/2016|EPI_ISL_335174', 'A/Perth/16/2009|EPI_ISL_31913', 'A/Brisbane/10/2007\xa0|EPI_ISL_177756', 'A/Wellington/1/2004|EPI_ISL_65217'])
285
149074
总序列数: 149383, 计划每份约: 24898 条
  - 已生成: /home

In [ ]:
# 三个合并起来,放到一个H3N2_all.fasta文件中
with open('/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2_all.fasta', 'w') as f:
    for name, seq in h3_68_03_seqs.items():
        f.write(f'>{name}\n')
        f.write(seq + '\n')
    for name, seq in h3_03_24_seqs.items():
        f.write(f'>{name}\n')
        f.write(seq + '\n')
    for name, seq in vcc_seqs.items():
        f.write(f'>{name}\n')
        f.write(seq + '\n')

# 将上面的这个文件划分成6等分,不要用seqbiot的划分工具,而是手动划分
all_seqs_dict = read_custom_fasta('/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2_all.fasta')
all_seqs_list = list(all_seqs_dict.items())
total_count = len(all_seqs_list)
split_num = 6

# 计算每一份的大小 (向上取整，确保最后一份不会剩下太多，或者用整除+余数逻辑)
import math
import os
chunk_size = math.ceil(total_count / split_num)

print(f"总序列数: {total_count}, 计划每份约: {chunk_size} 条")

for i in range(split_num):
    # 计算当前切片的起始和结束索引
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, total_count)
    
    # 获取当前分片的序列数据
    current_batch = all_seqs_list[start_idx:end_idx]
    
    # 如果已经没有数据了（针对不能整除的情况），跳出循环
    if not current_batch:
        break
        
    # 生成分片文件名，例如 H3N2_all_part1.fasta
    part_filename = os.path.join(f'/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2_all_part{i+1}.fasta')
    
    with open(part_filename, 'w') as f_part:
        for name, seq in current_batch:
            f_part.write(f'>{name}\n{seq}\n')
            
    print(f"  - 已生成: {part_filename} (包含 {len(current_batch)} 条序列)")

print("划分完成。")

In [6]:
from collections import defaultdict

def check_duplicates_content(path):
    # 使用 defaultdict(list) 来存储，结构为： { "名字": ["序列1", "序列2", ...] }
    records = defaultdict(list)
    
    print(f"正在读取文件: {path} ...")
    
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    
    # 1. 读取并归类所有序列
    for i in range(0, len(lines), 2):
        name = lines[i][1:]        # 去掉 >
        seq = lines[i + 1].upper() # 统一转大写
        records[name].append(seq)
        
    print(f"文件读取完毕，共解析出 {len(lines)//2} 条记录。\n")
    
    # 2. 分析重复项
    dup_same_seq_count = 0  # 名字相同，序列也相同
    dup_diff_seq_count = 0  # 名字相同，但序列不同
    
    duplicates = {k: v for k, v in records.items() if len(v) > 1}
    
    if not duplicates:
        print("完美！没有发现任何重复的名字。")
        return

    print(f"发现 {len(duplicates)} 个名字出现了多次。正在比对序列内容...\n")
    print(f"{'名字':<40} | {'重复次数':<10} | {'序列一致性检查'}")
    print("-" * 80)

    for name, seq_list in duplicates.items():
        count = len(seq_list)
        # 使用 set 来检查列表中的序列是否唯一
        unique_seqs = set(seq_list)
        
        if len(unique_seqs) == 1:
            status = "✅ 序列完全一致 (冗余)"
            dup_same_seq_count += 1
        else:
            status = "❌ 序列不一致 (冲突!)"
            dup_diff_seq_count += 1
            
        print(f"{name:<40} | {count:<10} | {status}")

    print("-" * 80)
    print("总结:")
    print(f"1. 名字重复且序列相同的数量: {dup_same_seq_count} (可以直接去重)")
    print(f"2. 名字重复但序列不同的数量: {dup_diff_seq_count} (必须重命名，否则会丢失数据)")

# 运行检查
file_path = '/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2.fasta'
check_duplicates_content(file_path)

正在读取文件: /home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2.fasta ...
文件读取完毕，共解析出 149091 条记录。

发现 17 个名字出现了多次。正在比对序列内容...

名字                                       | 重复次数       | 序列一致性检查
--------------------------------------------------------------------------------
A/Lusaka/04-NIC-450/2023|Original        | 2          | ✅ 序列完全一致 (冗余)
A/SouthAfrica/PET29353/2023|             | 2          | ✅ 序列完全一致 (冗余)
A/Chiang Rai/P2513/2023|Original         | 2          | ✅ 序列完全一致 (冗余)
A/Surat Thani/P2518/2023|Original        | 2          | ✅ 序列完全一致 (冗余)
A/Singapore/GP9370/2023|clinical specimen | 2          | ✅ 序列完全一致 (冗余)
A/South Africa/R07772/2023|OR_IR         | 2          | ✅ 序列完全一致 (冗余)
A/Sydney/443/2023|MDCK2                  | 2          | ✅ 序列完全一致 (冗余)
A/Victoria/1768/2023|MDCK1               | 2          | ✅ 序列完全一致 (冗余)
A/Kenya/AFI-ISB-1056/2023|               | 2          | ✅ 序列完全一致 (冗余)
A/Zambia/4450/2023|OR_

In [5]:
import re
# 读出序列和name
wrong_year_dict = {
    '0214':'2014',
    '2106':'2016',
    '2107':'2017',
}
def read_custom_fasta(path):
    # 键是year,值是name和seq
    year_name_seq = {}
    special_virus = []
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 2):
        name = lines[i][1:]
        seq = lines[i+1]

        # 提取年份
        base_part = name.split('|')[0]
        last_segment = base_part.split('/')[-1].strip()
        year = last_segment[:4]
        if year.isdigit() and len(year) == 4:
            year = wrong_year_dict.get(year, year)
        elif year.isdigit() and len(year) == 2:
            if int(year) >= 26:
                year = '19' + year
            else:
                year = '20' + year
        else:
            year = last_segment[:2]  
            if year.isdigit() and len(year) == 2:
                if int(year) >= 26:
                    year = '19' + year
                else:
                    year = '20' + year
                
            else:
                year = base_part.split('/')[-2].strip()
                if year.isdigit() and len(year) == 4:
                    pass
                else:
                    print(f"No year found in name: {name}")
                    special_virus.append((name, seq))
                    
        if int(year) not in year_name_seq:
            year_name_seq[int(year)] = {}
        year_name_seq[int(year)][lines[i].strip().replace('/', '')[1:]] = str(seq).strip().upper()
    return year_name_seq 

all_seq_dict = read_custom_fasta('/home/zhouchunyan/postgraduate/influenza-virus_LLM/research2_geometric_graph_learning/myResearch/data/raw/dataset0606/all_seq/H3N2_all.fasta')
print(sorted(all_seq_dict.keys()))
for key in sorted(all_seq_dict.keys()):
    print(f'{key}: {len(all_seq_dict[key])}')

[1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
1968: 4
1969: 3
1970: 2
1971: 4
1972: 5
1973: 4
1974: 5
1975: 7
1976: 7
1977: 7
1979: 1
1980: 2
1981: 2
1982: 6
1983: 1
1984: 3
1985: 5
1986: 2
1987: 3
1988: 6
1989: 17
1990: 7
1991: 19
1992: 49
1993: 44
1994: 10
1995: 17
1996: 10
1997: 14
1998: 4
1999: 412
2000: 404
2001: 288
2002: 759
2003: 1212
2004: 1119
2005: 1187
2006: 1081
2007: 1228
2008: 1642
2009: 1600
2010: 1696
2011: 2206
2012: 3609
2013: 2796
2014: 5584
2015: 6917
2016: 8120
2017: 15989
2018: 10756
2019: 14573
2020: 2828
2021: 6999
2022: 27711
2023: 14917
2024: 13473
2025: 2
